In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../../data/raw/paysim.csv")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nFraud distribution:")
print(df["isFraud"].value_counts())

print("\nFraud percentage:")
print(df["isFraud"].value_counts(normalize=True) * 100)

Shape: (6362620, 11)

Columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Data types:
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Duplicate rows: 0

Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud percentage:
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [2]:
# 1. Duplicate rows
print("Duplicate rows:", df.duplicated().sum())


# 2. Target distribution
print("\nFraud distribution:")
print(df["isFraud"].value_counts())

print("\nFraud percentage:")
print((df["isFraud"].value_counts(normalize=True) * 100).round(4))


# 3. Transaction types
print("\nTransaction types:")
print(df["type"].value_counts())


# 4. Unique values
print("\nUnique values:")
print(df.nunique())


# 5. Numerical summary
print("\nNumerical summary:")
print(df.describe().T)

Duplicate rows: 0

Fraud distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Fraud percentage:
isFraud
0    99.8709
1     0.1291
Name: proportion, dtype: float64

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

Unique values:
step                  743
type                    5
amount            5316900
nameOrig          6353307
oldbalanceOrg     1845844
newbalanceOrig    2682586
nameDest          2722362
oldbalanceDest    3614697
newbalanceDest    3555499
isFraud                 2
isFlaggedFraud          2
dtype: int64

Numerical summary:
                    count          mean           std  min       25%  \
step            6362620.0  2.433972e+02  1.423320e+02  1.0    156.00   
amount          6362620.0  1.798619e+05  6.038582e+05  0.0  13389.57   
oldbalanceOrg   6362620.0  8.338831e+05  2.888243e+06  0.0      0.00   
newbalanceOrig  6362620.0  8.551137e+05  

In [3]:
# Compare numerical features for fraud vs normal transactions

numeric_cols = [
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest",
    "isFlaggedFraud"
]

print(df.groupby("isFraud")[numeric_cols].mean().T)

isFraud                    0             1
amount          1.781970e+05  1.467967e+06
oldbalanceOrg   8.328287e+05  1.649668e+06
newbalanceOrig  8.559702e+05  1.923926e+05
oldbalanceDest  1.101421e+06  5.442496e+05
newbalanceDest  1.224926e+06  1.279708e+06
isFlaggedFraud  0.000000e+00  1.948131e-03


In [4]:
# Fraud transactions by transaction type

fraud_by_type = pd.crosstab(
    df["type"],
    df["isFraud"],
    normalize="index"
) * 100

print(fraud_by_type.round(4))

isFraud          0       1
type                      
CASH_IN   100.0000  0.0000
CASH_OUT   99.8160  0.1840
DEBIT     100.0000  0.0000
PAYMENT   100.0000  0.0000
TRANSFER   99.2312  0.7688


In [5]:
# Create a copy so original dataset remains unchanged
df_processed = df.copy()

# 1. Change in origin account balance
df_processed["balance_change_orig"] = (
    df_processed["oldbalanceOrg"] -
    df_processed["newbalanceOrig"]
)

# 2. Change in destination account balance
df_processed["balance_change_dest"] = (
    df_processed["newbalanceDest"] -
    df_processed["oldbalanceDest"]
)

# 3. Difference between transaction amount
# and actual change in origin balance
df_processed["orig_balance_error"] = (
    df_processed["amount"] -
    df_processed["balance_change_orig"]
)

# 4. Difference between transaction amount
# and actual change in destination balance
df_processed["dest_balance_error"] = (
    df_processed["amount"] -
    df_processed["balance_change_dest"]
)

print("New shape:", df_processed.shape)

print("\nNew columns:")
print(df_processed.columns.tolist())

New shape: (6362620, 15)

New columns:
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'balance_change_orig', 'balance_change_dest', 'orig_balance_error', 'dest_balance_error']


In [6]:
# Drop raw account ID columns
df_processed = df_processed.drop(
    columns=["nameOrig", "nameDest"]
)

print("Shape after dropping IDs:", df_processed.shape)
print(df_processed.columns.tolist())

Shape after dropping IDs: (6362620, 13)
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'balance_change_orig', 'balance_change_dest', 'orig_balance_error', 'dest_balance_error']


In [7]:
df_processed = pd.get_dummies(
    df_processed,
    columns=["type"],
    dtype=int
)

print("Shape after encoding:", df_processed.shape)
print(df_processed.head())

Shape after encoding: (6362620, 17)
   step    amount  oldbalanceOrg  newbalanceOrig  oldbalanceDest  \
0     1   9839.64       170136.0       160296.36             0.0   
1     1   1864.28        21249.0        19384.72             0.0   
2     1    181.00          181.0            0.00             0.0   
3     1    181.00          181.0            0.00         21182.0   
4     1  11668.14        41554.0        29885.86             0.0   

   newbalanceDest  isFraud  isFlaggedFraud  balance_change_orig  \
0             0.0        0               0              9839.64   
1             0.0        0               0              1864.28   
2             0.0        1               0               181.00   
3             0.0        1               0               181.00   
4             0.0        0               0             11668.14   

   balance_change_dest  orig_balance_error  dest_balance_error  type_CASH_IN  \
0                  0.0       -1.455192e-11             9839.64          

In [8]:
# Separate features and target

X = df_processed.drop(columns=["isFraud"])
y = df_processed["isFraud"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (6362620, 16)
y shape: (6362620,)

Target distribution:
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining fraud distribution:")
print(y_train.value_counts())

print("\nTesting fraud distribution:")
print(y_test.value_counts())

Training shape: (5090096, 16)
Testing shape: (1272524, 16)

Training fraud distribution:
isFraud
0    5083526
1       6570
Name: count, dtype: int64

Testing fraud distribution:
isFraud
0    1270881
1       1643
Name: count, dtype: int64


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (5090096, 16)
X_test_scaled shape: (1272524, 16)


In [11]:
print("Before balancing:")

print("\nTraining:")
print(y_train.value_counts())

print("\nTesting:")
print(y_test.value_counts())

print("\nTraining percentage:")
print(
    (y_train.value_counts(normalize=True) * 100).round(4)
)

Before balancing:

Training:
isFraud
0    5083526
1       6570
Name: count, dtype: int64

Testing:
isFraud
0    1270881
1       1643
Name: count, dtype: int64

Training percentage:
isFraud
0    99.8709
1     0.1291
Name: proportion, dtype: float64


In [12]:
print("Infinite values in X_train:",
      np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())

print("Infinite values in X_test:",
      np.isinf(X_test.select_dtypes(include=np.number)).sum().sum())

print("NaN values in X_train:",
      X_train.isna().sum().sum())

print("NaN values in X_test:",
      X_test.isna().sum().sum())

Infinite values in X_train: 0
Infinite values in X_test: 0
NaN values in X_train: 0
NaN values in X_test: 0


In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model.fit(X_train_scaled, y_train)

print("Model training completed!")

Model training completed!


In [14]:
y_pred = model.predict(X_test_scaled)

print("Predictions completed!")

Predictions completed!


In [15]:
from sklearn.metrics import classification_report, confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[1209643   61238]
 [     32    1611]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.95      0.98   1270881
           1       0.03      0.98      0.05      1643

    accuracy                           0.95   1272524
   macro avg       0.51      0.97      0.51   1272524
weighted avg       1.00      0.95      0.97   1272524



In [16]:
# Get probability of fraud (class 1)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(y_prob[:10])

[1.45606057e-12 1.32624069e-08 1.68193915e-06 2.67392131e-03
 6.96604844e-02 5.15663461e-01 4.98440586e-01 1.78701804e-03
 7.17036223e-02 5.49347730e-07]


In [17]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold, zero_division=0)
    recall = recall_score(y_test, y_pred_threshold, zero_division=0)
    f1 = f1_score(y_test, y_pred_threshold, zero_division=0)

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f}"
    )

Threshold: 0.1 | Precision: 0.009 | Recall: 0.999 | F1: 0.019
Threshold: 0.2 | Precision: 0.012 | Recall: 0.999 | F1: 0.024
Threshold: 0.3 | Precision: 0.015 | Recall: 0.998 | F1: 0.029
Threshold: 0.4 | Precision: 0.019 | Recall: 0.993 | F1: 0.037
Threshold: 0.5 | Precision: 0.026 | Recall: 0.981 | F1: 0.050
Threshold: 0.6 | Precision: 0.036 | Recall: 0.967 | F1: 0.070
Threshold: 0.7 | Precision: 0.051 | Recall: 0.937 | F1: 0.097
Threshold: 0.8 | Precision: 0.076 | Recall: 0.899 | F1: 0.140
Threshold: 0.9 | Precision: 0.136 | Recall: 0.821 | F1: 0.233


In [18]:
print(X_train.columns.tolist())

['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFlaggedFraud', 'balance_change_orig', 'balance_change_dest', 'orig_balance_error', 'dest_balance_error', 'type_CASH_IN', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [19]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest training completed!")

Random Forest training completed!


In [20]:
# Make predictions using Random Forest

y_pred_rf = rf_model.predict(X_test)

print("Predictions completed!")

Predictions completed!


In [21]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf, zero_division=0))
print("Recall:", recall_score(y_test, y_pred_rf, zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred_rf, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

Accuracy: 0.9999968566408177
Precision: 1.0
Recall: 0.9975654290931223
F1 Score: 0.9987812309567337

Confusion Matrix:
[[1270881       0]
 [      4    1639]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524



In [22]:
import pandas as pd

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

                Feature  Importance
9    orig_balance_error    0.354687
7   balance_change_orig    0.227063
3        newbalanceOrig    0.128010
2         oldbalanceOrg    0.088114
1                amount    0.048513
14         type_PAYMENT    0.039331
0                  step    0.022546
8   balance_change_dest    0.019196
15        type_TRANSFER    0.018594
12        type_CASH_OUT    0.014230
10   dest_balance_error    0.014011
11         type_CASH_IN    0.012091
5        newbalanceDest    0.006349
4        oldbalanceDest    0.005238
6        isFlaggedFraud    0.001515
13           type_DEBIT    0.000512


In [23]:
# Remove engineered features for comparison

engineered_features = [
    "balance_change_orig",
    "balance_change_dest",
    "orig_balance_error",
    "dest_balance_error"
]

X_train_original = X_train.drop(columns=engineered_features)
X_test_original = X_test.drop(columns=engineered_features)

print("Original features:", X_train_original.shape)

Original features: (5090096, 12)


In [24]:
rf_original = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_original.fit(X_train_original, y_train)

print("Training completed!")

Training completed!


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_original = rf_original.predict(X_test_original)

print("Accuracy:", accuracy_score(y_test, y_pred_original))
print("Precision:", precision_score(y_test, y_pred_original))
print("Recall:", recall_score(y_test, y_pred_original))
print("F1 Score:", f1_score(y_test, y_pred_original))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_original))

Accuracy: 0.9996943083195288
Precision: 0.9815668202764977
Recall: 0.7778454047474133
F1 Score: 0.867911714770798

Confusion Matrix:
[[1270857      24]
 [    365    1278]]


In [26]:
# Get fraud probability from the better Random Forest
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print(rf_prob[:10])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [27]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

for threshold in thresholds:

    y_pred_rf = (rf_prob >= threshold).astype(int)

    precision = precision_score(
        y_test, y_pred_rf, zero_division=0
    )

    recall = recall_score(
        y_test, y_pred_rf, zero_division=0
    )

    f1 = f1_score(
        y_test, y_pred_rf, zero_division=0
    )

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} | "
        f"F1: {f1:.4f}"
    )

Threshold: 0.10 | Precision: 0.9939 | Recall: 0.9976 | F1: 0.9957
Threshold: 0.20 | Precision: 1.0000 | Recall: 0.9976 | F1: 0.9988
Threshold: 0.30 | Precision: 1.0000 | Recall: 0.9976 | F1: 0.9988
Threshold: 0.40 | Precision: 1.0000 | Recall: 0.9976 | F1: 0.9988
Threshold: 0.50 | Precision: 1.0000 | Recall: 0.9976 | F1: 0.9988
Threshold: 0.60 | Precision: 1.0000 | Recall: 0.9976 | F1: 0.9988
Threshold: 0.70 | Precision: 1.0000 | Recall: 0.9976 | F1: 0.9988
Threshold: 0.80 | Precision: 1.0000 | Recall: 0.9921 | F1: 0.9960
Threshold: 0.90 | Precision: 1.0000 | Recall: 0.9884 | F1: 0.9942


In [28]:
import joblib
from pathlib import Path

MODEL_DIR = Path("../model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save trained Random Forest
joblib.dump(rf_model, MODEL_DIR / "fraud_model.pkl")

# Save the exact feature order used during training
joblib.dump(list(X_train.columns), MODEL_DIR / "feature_columns.pkl")

# Save final threshold
joblib.dump(0.50, MODEL_DIR / "threshold.pkl")

# Save feature importance
joblib.dump(
    feature_importance,
    MODEL_DIR / "feature_importance.pkl"
)

print("Model files saved successfully!")

Model files saved successfully!


In [29]:
# Export raw test/train CSVs for the live-stream replay script
# (This is the missing step — model ko save kiya tha, lekin CSVs kahin save nahi hui thi)

RAW_COLUMNS = [
    "step", "type", "amount", "nameOrig", "oldbalanceOrg", "newbalanceOrig",
    "nameDest", "oldbalanceDest", "newbalanceDest", "isFraud"
]

# IMPORTANT FIX: tumhari PROCESSED_DIR "../data/processed" thi jo galat
# folder point karti hai (notebook ml/notebooks/ mein hai, isliye
# "../data" ml/data ban jaata — jabki asli project ka data/ folder
# do level upar hai). Ye line use sahi karti hai:
from pathlib import Path
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_raw = df.loc[X_train.index, RAW_COLUMNS].sort_values("step").reset_index(drop=True)
test_raw = df.loc[X_test.index, RAW_COLUMNS].sort_values("step").reset_index(drop=True)

train_raw.to_csv(PROCESSED_DIR / "train.csv", index=False)
test_raw.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("Saved:")
print(" -", (PROCESSED_DIR / "train.csv").resolve())
print(" -", (PROCESSED_DIR / "test.csv").resolve())
print("\ntest.csv shape:", test_raw.shape, "| fraud rows:", test_raw["isFraud"].sum())

Saved:
 - C:\Users\joysh\OneDrive\Desktop\FraudLens\data\processed\train.csv
 - C:\Users\joysh\OneDrive\Desktop\FraudLens\data\processed\test.csv

test.csv shape: (1272524, 10) | fraud rows: 1643
